In [2]:
print("test")

test


In [1]:
%pip install pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 55.2 MB/s  0:00:00m0:00:0100:01

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
!lscpu

Architecture:                x86_64
  CPU op-mode(s):            32-bit, 64-bit
  Address sizes:             46 bits physical, 57 bits virtual
  Byte Order:                Little Endian
CPU(s):                      16
  On-line CPU(s) list:       0-15
Vendor ID:                   GenuineIntel
  Model name:                Intel(R) Xeon(R) Platinum 8370C CPU @ 2.80GHz
    CPU family:              6
    Model:                   106
    Thread(s) per core:      2
    Core(s) per socket:      8
    Socket(s):               1
    Stepping:                6
    CPU(s) scaling MHz:      99%
    CPU max MHz:             2800.0000
    CPU min MHz:             800.0000
    BogoMIPS:                5586.87
    Flags:                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pg
                             e mca cmov pat pse36 clflush mmx fxsr sse sse2 ss h
                             t syscall nx pdpe1gb rdtscp lm constant_tsc rep_goo
                             d nopl xtopology tsc_rel

In [3]:
%pip install pandas



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
import pandas as pd

df = pd.read_csv("../../aew-data/test-blob/input_data/HackDays2026 - GIGI.csv", sep=";")

df.head()



,GP-Nr,PLZ,Ort,Kanton,WärmePumpe,PV,PV-Leistung in kWp,Batterie/Speicher,Ladestation für Elektrofahrzeuge,Wärmepumpenboiler,Datum Unterschrift,geplanter Baustart,Übergabe,InBetrieb-Datum
0,698970,5452.0,Oberrohrdorf,AG,-,x,NaN,-,-,-,18.04.2008,NaN,NaN,NaN
1,NaN,5024.0,Küttigen,AG,-,-,NaN,-,-,-,01.01.2017,NaN,NaN,NaN
2,196344,5732.0,Zetzwil,AG,-,-,NaN,-,-,-,01.01.2017,NaN,NaN,21.06.2017
3,570021,5023.0,Biberstein,AG,x,-,NaN,-,-,-,01.07.2017,NaN,NaN,24.11.2017
4,716671,8962.0,Bergdietikon,AG,x,x,NaN,x,-,-,24.08.2017,NaN,23.11.2017,04.12.2017


/home/renku/work/OSNOVA/dataanalysis


In [ ]:
from pathlib import Path
import pandas as pd

source = Path("../../aew-data/test-blob/input_data")
target = Path("../../store/parquet_data")

for csv_file in source.rglob("*.csv"):
    relative = csv_file.relative_to(source)
    parquet_file = target / relative.with_suffix(".parquet")

    parquet_file.parent.mkdir(parents=True, exist_ok=True)

    try:
        # Your GIGI file uses ;
        df = pd.read_csv(csv_file, sep=";")

        # Clean column names
        df.columns = df.columns.str.strip()

        df.to_parquet(
            parquet_file,
            engine="pyarrow",
            compression="snappy",
            index=False
        )

        print(f"Converted: {csv_file} -> {parquet_file}")

    except Exception as e:
        print(f"FAILED: {csv_file}")
        print(e)

Converted: ../../aew-data/test-blob/input_data/HackDays2026 - GIGI.csv -> ../../store/parquet_data/HackDays2026 - GIGI.parquet
Converted: ../../aew-data/test-blob/input_data/Zähler-GP.csv -> ../../store/parquet_data/Zähler-GP.parquet
Converted: ../../aew-data/test-blob/input_data/mpid_zähler_mapping.csv -> ../../store/parquet_data/mpid_zähler_mapping.parquet
Converted: ../../aew-data/test-blob/input_data/2026/April 2026/LG_AIM2Hackerdays_kWh_20260728_140238.csv -> ../../store/parquet_data/2026/April 2026/LG_AIM2Hackerdays_kWh_20260728_140238.parquet
Converted: ../../aew-data/test-blob/input_data/2026/Februar 2026/LG_AIM2Hackerdays_kWh_20260727_130937.csv -> ../../store/parquet_data/2026/Februar 2026/LG_AIM2Hackerdays_kWh_20260727_130937.parquet
Converted: ../../aew-data/test-blob/input_data/2026/Januar 2026/LG_AIM2Hackerdays_kWh_20260727_065429.csv -> ../../store/parquet_data/2026/Januar 2026/LG_AIM2Hackerdays_kWh_20260727_065429.parquet
Converted: ../../aew-data/test-blob/input_data/2

In [ ]:
from pathlib import Path
import pandas as pd

source = Path("../../aew-data/test-blob/input_data")
target = Path("../../store/parquet_data")

skip_years = {}

for csv_file in source.rglob("*.csv"):
    relative = csv_file.relative_to(source)

    # Skip anything inside 2025 or 2026 folders
    if any(part in skip_years for part in relative.parts):
        print(f"SKIPPED YEAR: {csv_file}")
        continue

    parquet_file = target / relative.with_suffix(".parquet")
    parquet_file.parent.mkdir(parents=True, exist_ok=True)

    # Skip files that were already converted
    if parquet_file.exists():
        print(f"ALREADY EXISTS: {parquet_file}")
        continue

    try:
        df = pd.read_csv(csv_file, sep=";")
        df.columns = df.columns.str.strip()

        df.to_parquet(
            parquet_file,
            engine="pyarrow",
            compression="snappy",
            index=False
        )

        print(f"Converted: {csv_file} -> {parquet_file}")

    except Exception as e:
        print(f"FAILED: {csv_file}")
        print(e)

print("done")

ALREADY EXISTS: ../../store/parquet_data/HackDays2026 - GIGI.parquet
ALREADY EXISTS: ../../store/parquet_data/Zähler-GP.parquet
ALREADY EXISTS: ../../store/parquet_data/mpid_zähler_mapping.parquet
SKIPPED YEAR: ../../aew-data/test-blob/input_data/2026/April 2026/LG_AIM2Hackerdays_kWh_20260728_140238.csv
SKIPPED YEAR: ../../aew-data/test-blob/input_data/2026/Februar 2026/LG_AIM2Hackerdays_kWh_20260727_130937.csv
SKIPPED YEAR: ../../aew-data/test-blob/input_data/2026/Januar 2026/LG_AIM2Hackerdays_kWh_20260727_065429.csv
SKIPPED YEAR: ../../aew-data/test-blob/input_data/2026/Juli 2026/LG_AIM2Hackerdays_kWh_20260824_102023.csv
SKIPPED YEAR: ../../aew-data/test-blob/input_data/2026/Juni 2026/LG_AIM2Hackerdays_kWh_20260729_062053.csv
SKIPPED YEAR: ../../aew-data/test-blob/input_data/2026/Mai 2026/LG_AIM2Hackerdays_kWh_20260728_195653.csv
SKIPPED YEAR: ../../aew-data/test-blob/input_data/2026/März 2026/LG_AIM2Hackerdays_kWh_20260728_063802.csv
SKIPPED YEAR: ../../aew-data/test-blob/input_data

In [4]:
from pathlib import Path
import pandas as pd
import csv

source = Path("../../aew-data/test-blob/input_data")
target = Path("../../store/parquet_data")

wanted_years = {"2023", "2024"}

for csv_file in source.rglob("*.csv"):
    relative = csv_file.relative_to(source)

    # Only process 2023 and 2024
    year = next(
        (part for part in relative.parts if part in wanted_years),
        None
    )

    if year is None:
        print(f"SKIPPED: {csv_file}")
        continue

    parquet_file = target / relative.with_suffix(".parquet")
    parquet_file.parent.mkdir(parents=True, exist_ok=True)

    # 2024 files can be skipped if already converted.
    # 2023 files are intentionally overwritten because we need to fix them.
    if year == "2024" and parquet_file.exists():
        print(f"ALREADY EXISTS: {parquet_file}")
        continue

    try:
        if year == "2023":
            # Read header ourselves
            with open(csv_file, "r", encoding="utf-8-sig") as f:
                reader = csv.reader(f, delimiter=";")
                header = next(reader)

            # Remove trailing empty header caused by final ;
            if header and header[-1] == "":
                header = header[:-1]

            # Read data without trusting the malformed header
            df = pd.read_csv(
                csv_file,
                sep=";",
                header=None,
                skiprows=1
            )

            # Remove trailing empty column caused by final ;
            if df.iloc[:, -1].isna().all():
                df = df.iloc[:, :-1]

            # 2023 has an extra second column.
            # Remove it.
            df = df.drop(columns=df.columns[1])

            # Verify that everything now lines up
            if len(df.columns) != len(header):
                raise ValueError(
                    f"Column mismatch after 2023 fix: "
                    f"{len(df.columns)} data columns vs "
                    f"{len(header)} header columns"
                )

            df.columns = header

            print(
                f"2023 FIX: removed extra identifier column "
                f"from {csv_file.name}"
            )

        else:
            # Normal 2024 structure
            df = pd.read_csv(csv_file, sep=";")

            # Remove completely empty trailing column, if present
            unnamed = [
                col for col in df.columns
                if str(col).startswith("Unnamed:")
            ]

            if unnamed:
                df = df.drop(columns=unnamed)

        # Clean whitespace from column names
        df.columns = df.columns.str.strip()

        df.to_parquet(
            parquet_file,
            engine="pyarrow",
            compression="snappy",
            index=False
        )

        print(f"Converted: {csv_file} -> {parquet_file}")

    except Exception as e:
        print(f"FAILED: {csv_file}")
        print(e)

SKIPPED: ../../aew-data/test-blob/input_data/HackDays2026 - GIGI.csv
SKIPPED: ../../aew-data/test-blob/input_data/Zähler-GP.csv
SKIPPED: ../../aew-data/test-blob/input_data/mpid_zähler_mapping.csv
SKIPPED: ../../aew-data/test-blob/input_data/2026/April 2026/LG_AIM2Hackerdays_kWh_20260728_140238.csv
SKIPPED: ../../aew-data/test-blob/input_data/2026/Februar 2026/LG_AIM2Hackerdays_kWh_20260727_130937.csv
SKIPPED: ../../aew-data/test-blob/input_data/2026/Januar 2026/LG_AIM2Hackerdays_kWh_20260727_065429.csv
SKIPPED: ../../aew-data/test-blob/input_data/2026/Juli 2026/LG_AIM2Hackerdays_kWh_20260824_102023.csv
SKIPPED: ../../aew-data/test-blob/input_data/2026/Juni 2026/LG_AIM2Hackerdays_kWh_20260729_062053.csv
SKIPPED: ../../aew-data/test-blob/input_data/2026/Mai 2026/LG_AIM2Hackerdays_kWh_20260728_195653.csv
SKIPPED: ../../aew-data/test-blob/input_data/2026/März 2026/LG_AIM2Hackerdays_kWh_20260728_063802.csv
SKIPPED: ../../aew-data/test-blob/input_data/2025/April 2025/LG_AIM2Hackerdays_kWh_2

In [2]:
from pathlib import Path
import pandas as pd
import re

root = Path("../../store/parquet_data")

negative_rows = []
file_summary = []

# Matches columns such as 00:15, 13:30, 23:45, 00:00
time_pattern = re.compile(r"^\d{2}:\d{2}$")

for parquet_file in root.rglob("*.parquet"):
    try:
        df = pd.read_parquet(parquet_file)

        # Find only quarter-hour measurement columns
        time_cols = [
            col for col in df.columns
            if time_pattern.match(str(col))
        ]

        if not time_cols:
            print(f"SKIPPED (no time columns): {parquet_file}")
            continue

        # Ensure values are numeric
        values = df[time_cols].apply(pd.to_numeric, errors="coerce")

        # Boolean matrix: True wherever value < 0
        negative_mask = values < 0
        negative_count = int(negative_mask.sum().sum())

        file_summary.append({
            "file": str(parquet_file),
            "rows": len(df),
            "negative_values": negative_count
        })

        if negative_count > 0:
            print(f"NEGATIVE: {negative_count:>6} values in {parquet_file}")

            # Get positions of every negative value
            row_indices, col_indices = negative_mask.to_numpy().nonzero()

            for row_idx, col_idx in zip(row_indices, col_indices):
                time_col = time_cols[col_idx]

                negative_rows.append({
                    "file": str(parquet_file),
                    "MP ID": df.iloc[row_idx].get("MP ID"),
                    "OBIS-Code": df.iloc[row_idx].get("OBIS-Code"),
                    "Datum": df.iloc[row_idx].get("Datum"),
                    "PLZ": df.iloc[row_idx].get("PLZ"),
                    "time": time_col,
                    "value_kWh": values.iloc[row_idx, col_idx]
                })

        else:
            print(f"OK: {parquet_file}")

    except Exception as e:
        print(f"FAILED: {parquet_file}")
        print(e)

# Results
summary = pd.DataFrame(file_summary)
negatives = pd.DataFrame(negative_rows)

print("\n=============================")
print("TOTAL NEGATIVE VALUES:", len(negatives))
print("=============================")

negatives

SKIPPED (no time columns): ../../store/parquet_data/HackDays2026 - GIGI.parquet
SKIPPED (no time columns): ../../store/parquet_data/mpid_zähler_mapping.parquet
SKIPPED (no time columns): ../../store/parquet_data/Zähler-GP.parquet
NEGATIVE:      4 values in ../../store/parquet_data/2026/April 2026/LG_AIM2Hackerdays_kWh_20260728_140238.parquet
NEGATIVE:      1 values in ../../store/parquet_data/2026/Februar 2026/LG_AIM2Hackerdays_kWh_20260727_130937.parquet
NEGATIVE:      2 values in ../../store/parquet_data/2026/Januar 2026/LG_AIM2Hackerdays_kWh_20260727_065429.parquet
NEGATIVE:      3 values in ../../store/parquet_data/2026/Juli 2026/LG_AIM2Hackerdays_kWh_20260824_102023.parquet
NEGATIVE:      1 values in ../../store/parquet_data/2026/Juni 2026/LG_AIM2Hackerdays_kWh_20260729_062053.parquet
NEGATIVE:     13 values in ../../store/parquet_data/2026/Mai 2026/LG_AIM2Hackerdays_kWh_20260728_195653.parquet
NEGATIVE:      2 values in ../../store/parquet_data/2026/März 2026/LG_AIM2Hackerdays_kW

,file,MP ID,OBIS-Code,Datum,PLZ,time,value_kWh
0,../../store/parquet_data/2026/April 2026/LG_AI...,95183,1-1:1.29.0*255,23.04.2026,5304.0,04:15,-0.017
1,../../store/parquet_data/2026/April 2026/LG_AI...,166902,1-1:1.29.0*255,28.04.2026,5236.0,01:15,-16777.216
2,../../store/parquet_data/2026/April 2026/LG_AI...,166902,1-1:2.29.0*255,29.04.2026,5236.0,20:15,-0.256
3,../../store/parquet_data/2026/April 2026/LG_AI...,166902,1-1:2.29.0*255,29.04.2026,5236.0,20:45,-16777.216
4,../../store/parquet_data/2026/Februar 2026/LG_...,105491,1-1:1.29.0*255,28.02.2026,5504.0,04:15,-2.615
...,...,...,...,...,...,...,...
2132,../../store/parquet_data/2023/2023/September 2...,61016,1-1:1.29.0*255,04.09.2023,4310.0,05:00,-0.314
2133,../../store/parquet_data/2023/2023/September 2...,31024,1-1:1.29.0*255,30.09.2023,5726.0,23:45,-6.670
2134,../../store/parquet_data/2023/2023/September 2...,40887,1-1:1.29.0*255,02.09.2023,4310.0,10:30,-1.555
2135,../../store/parquet_data/2023/2023/September 2...,40887,1-1:1.29.0*255,12.09.2023,4310.0,11:00,-0.426


In [13]:
from pathlib import Path
import pandas as pd
import csv

source = Path("../../aew-data/test-blob/input_data")
target = Path("../../store/parquet_data")

for csv_file in source.rglob("*.csv"):
    relative = csv_file.relative_to(source)

    # Detect whether this file belongs to 2023
    year = next(
        (part for part in relative.parts if part.isdigit() and len(part) == 4),
        None
    )

    parquet_file = target / relative.with_suffix(".parquet")
    parquet_file.parent.mkdir(parents=True, exist_ok=True)

    try:
        if year == "2023":
            # Read header ourselves because the 2023 CSV structure is malformed
            with open(csv_file, "r", encoding="utf-8-sig") as f:
                reader = csv.reader(f, delimiter=";")
                header = next(reader)

            # Remove trailing empty header caused by a final ;
            if header and header[-1] == "":
                header = header[:-1]

            # Read the data without trusting the malformed header
            df = pd.read_csv(
                csv_file,
                sep=";",
                header=None,
                skiprows=1
            )

            # Remove trailing completely empty column caused by a final ;
            if len(df.columns) > 0 and df.iloc[:, -1].isna().all():
                df = df.iloc[:, :-1]

            # 2023 has an extra second column.
            # Remove it.
            if len(df.columns) < 2:
                raise ValueError(
                    f"2023 file has fewer than 2 columns: {csv_file}"
                )

            df = df.drop(columns=df.columns[1])

            # Verify that the corrected data matches the header
            if len(df.columns) != len(header):
                raise ValueError(
                    f"Column mismatch after 2023 fix: "
                    f"{len(df.columns)} data columns vs "
                    f"{len(header)} header columns "
                    f"in {csv_file}"
                )

            df.columns = header

            print(
                f"2023 FIX: removed extra second column "
                f"from {csv_file.name}"
            )

        else:
            # Normal structure for all other years
            df = pd.read_csv(
                csv_file,
                sep=";"
            )

            # Remove completely empty trailing columns created by a final ;
            unnamed = [
                col for col in df.columns
                if str(col).startswith("Unnamed:")
                and df[col].isna().all()
            ]

            if unnamed:
                df = df.drop(columns=unnamed)

        # Clean whitespace from column names
        df.columns = df.columns.astype(str).str.strip()

        # Always overwrite/reconvert the parquet file
        df.to_parquet(
            parquet_file,
            engine="pyarrow",
            compression="snappy",
            index=False
        )

        print(f"Converted: {csv_file} -> {parquet_file}")

    except Exception as e:
        print(f"FAILED: {csv_file}")
        print(f"{type(e).__name__}: {e}")

print("done")

Converted: ../../aew-data/test-blob/input_data/HackDays2026 - GIGI.csv -> ../../store/parquet_data/HackDays2026 - GIGI.parquet
Converted: ../../aew-data/test-blob/input_data/Zähler-GP.csv -> ../../store/parquet_data/Zähler-GP.parquet
Converted: ../../aew-data/test-blob/input_data/mpid_zähler_mapping.csv -> ../../store/parquet_data/mpid_zähler_mapping.parquet
Converted: ../../aew-data/test-blob/input_data/2026/April 2026/LG_AIM2Hackerdays_kWh_20260728_140238.csv -> ../../store/parquet_data/2026/April 2026/LG_AIM2Hackerdays_kWh_20260728_140238.parquet
Converted: ../../aew-data/test-blob/input_data/2026/Februar 2026/LG_AIM2Hackerdays_kWh_20260727_130937.csv -> ../../store/parquet_data/2026/Februar 2026/LG_AIM2Hackerdays_kWh_20260727_130937.parquet
Converted: ../../aew-data/test-blob/input_data/2026/Januar 2026/LG_AIM2Hackerdays_kWh_20260727_065429.csv -> ../../store/parquet_data/2026/Januar 2026/LG_AIM2Hackerdays_kWh_20260727_065429.parquet
Converted: ../../aew-data/test-blob/input_data/2

In [ ]:
negatives[]

,file,MP ID,OBIS-Code,Datum,PLZ,time,value_kWh
0,../../store/parquet_data/2026/April 2026/LG_AI...,95183,1-1:1.29.0*255,23.04.2026,5304.0,04:15,-0.017
1,../../store/parquet_data/2026/April 2026/LG_AI...,166902,1-1:1.29.0*255,28.04.2026,5236.0,01:15,-16777.216
2,../../store/parquet_data/2026/April 2026/LG_AI...,166902,1-1:2.29.0*255,29.04.2026,5236.0,20:15,-0.256
3,../../store/parquet_data/2026/April 2026/LG_AI...,166902,1-1:2.29.0*255,29.04.2026,5236.0,20:45,-16777.216
4,../../store/parquet_data/2026/Februar 2026/LG_...,105491,1-1:1.29.0*255,28.02.2026,5504.0,04:15,-2.615
...,...,...,...,...,...,...,...
2132,../../store/parquet_data/2023/2023/September 2...,61016,1-1:1.29.0*255,04.09.2023,4310.0,05:00,-0.314
2133,../../store/parquet_data/2023/2023/September 2...,31024,1-1:1.29.0*255,30.09.2023,5726.0,23:45,-6.670
2134,../../store/parquet_data/2023/2023/September 2...,40887,1-1:1.29.0*255,02.09.2023,4310.0,10:30,-1.555
2135,../../store/parquet_data/2023/2023/September 2...,40887,1-1:1.29.0*255,12.09.2023,4310.0,11:00,-0.426


In [4]:
unique_mpids = negatives["MP ID"].unique()
unique_mpids

array([np.int64(95183), np.int64(166902), np.int64(105491),
       np.int64(65532), np.int64(60854), np.int64(29230),
       np.int64(163761), np.int64(484), np.int64(168419),
       np.int64(165291), np.int64(49554), np.int64(156396),
       np.int64(109577), np.int64(100196), np.int64(38459),
       np.int64(121704), np.int64(31024), np.int64(57344),
       np.int64(32769), np.int64(30223), np.int64(84020), np.int64(99753),
       np.int64(111270), np.int64(54312), np.int64(73323),
       np.int64(81567), np.int64(37862), np.int64(119217),
       np.int64(124902), np.int64(122184), np.int64(95396),
       np.int64(88023), np.int64(138629), np.int64(111623),
       np.int64(11406), np.int64(134166), np.int64(60475),
       np.int64(106193), np.int64(113715), np.int64(76527),
       np.int64(104725), np.int64(49289), np.int64(52799),
       np.int64(61139), np.int64(138226), np.int64(96989),
       np.int64(60033), np.int64(130205), np.int64(7412), np.int64(67346),
       np.int64(4088

In [5]:
import pandas as pd

# --------------------------------------------------
# 1. Load the three lookup tables
# --------------------------------------------------

mpid_map = pd.read_parquet(
    "../../store/parquet_data/mpid_zähler_mapping.parquet"
)

zaehler_gp = pd.read_parquet(
    "../../store/parquet_data/Zähler-GP.parquet"
)

gigi = pd.read_parquet(
    "../../store/parquet_data/HackDays2026 - GIGI.parquet"
)

# Clean column names
mpid_map.columns = mpid_map.columns.str.strip()
zaehler_gp.columns = zaehler_gp.columns.str.strip()
gigi.columns = gigi.columns.str.strip()


# --------------------------------------------------
# 2. Normalize IDs so joins work reliably
# --------------------------------------------------

mpid_map["MP ID"] = pd.to_numeric(
    mpid_map["MP ID"],
    errors="coerce"
).astype("Int64")

zaehler_gp["GPartner"] = pd.to_numeric(
    zaehler_gp["GPartner"],
    errors="coerce"
).astype("Int64")

gigi["GP-Nr"] = pd.to_numeric(
    gigi["GP-Nr"],
    errors="coerce"
).astype("Int64")


# --------------------------------------------------
# 3. Convert GIGI PV field into clean labels
# --------------------------------------------------

gigi["PV"] = (
    gigi["PV"]
    .astype(str)
    .str.strip()
    .str.lower()
)

gigi["has_pv"] = gigi["PV"].map({
    "x": True,
    "-": False
})


# --------------------------------------------------
# 4. Start ONLY with the unique MP IDs you found
#    in the negative-reading dataset
# --------------------------------------------------

mpids = pd.DataFrame({
    "MP ID": pd.Series(unique_mpids, dtype="Int64")
})


# --------------------------------------------------
# 5. MP ID -> Zählpunktbezeichnung
# --------------------------------------------------

matched = mpids.merge(
    mpid_map[["MP ID", "Zählpunktbezeichnung"]],
    on="MP ID",
    how="left"
)


# --------------------------------------------------
# 6. Zählpunktbezeichnung -> GPartner
# --------------------------------------------------

matched = matched.merge(
    zaehler_gp[
        ["Zählpunktbezeichnung", "GPartner"]
    ],
    on="Zählpunktbezeichnung",
    how="left"
)


# --------------------------------------------------
# 7. GPartner -> GP-Nr in GIGI -> PV
# --------------------------------------------------

matched = matched.merge(
    gigi[["GP-Nr", "PV", "has_pv"]],
    left_on="GPartner",
    right_on="GP-Nr",
    how="left"
)

matched.head()

ValueError: invalid literal for int() with base 10: 'CH1011701234500000000000000349983'

In [6]:
clean_mpids = (
    pd.to_numeric(
        pd.Series(unique_mpids),
        errors="coerce"
    )
    .dropna()
    .astype("Int64")
    .drop_duplicates()
)

mpids = pd.DataFrame({
    "MP ID": clean_mpids
})

print("Original unique values:", len(unique_mpids))
print("Valid numeric MP IDs:", len(mpids))

Original unique values: 179
Valid numeric MP IDs: 128


In [7]:
all_mpids = pd.Series(unique_mpids)

bad_values = all_mpids[
    pd.to_numeric(all_mpids, errors="coerce").isna()
].drop_duplicates()

print("Bad/non-numeric values:", len(bad_values))
print(bad_values.tolist())

Bad/non-numeric values: 51
['CH1011701234500000000000000349983', 'CH1011701234500000000000000507844', 'CH1011701234500000000000002125044', 'CH1011701234500000000000000512076', 'CH1011701234500000000000000527602', 'CH1011701234500000000000000541561', 'CH1011701234500000000000000557546', 'CH1011701234500000000000002099327', 'CH1011701234500000000000002099404', 'CH1011701234500000000000000578168', 'CH1011701234500000000000000600934', 'CH1011701234500000000000000535216', 'CH1011701234500000000000000536910', 'CH1011701234500000000000000546300', 'CH1011701234500000000000000548647', 'CH1011701234500000000000000553425', 'CH1011701234500000000000000574435', 'CH1011701234500000000000000580579', 'CH1011701234500000000000000599230', 'CH1011701234500000000000002094042', 'CH1011701234500000000000000525294', 'CH1011701234500000000000000527464', 'CH1011701234500000000000000552805', 'CH1011701234500000000000000575709', 'CH1011701234500000000000000604644', 'CH1011701234500000000000000616322', 'CH1011701

In [8]:
clean_mpids = (
    pd.to_numeric(
        pd.Series(unique_mpids),
        errors="coerce"
    )
    .dropna()
    .astype("Int64")
    .drop_duplicates()
)

mpids = pd.DataFrame({
    "MP ID": clean_mpids
})


In [9]:
bad = pd.to_numeric(
    negatives["MP ID"],
    errors="coerce"
).isna()

print("Invalid MP IDs:", bad.sum())

Invalid MP IDs: 194


In [10]:
from pathlib import Path
import pandas as pd

root = Path("../../store/parquet_data")

dates = []

for year in ["2023", "2026"]:
    for file in (root / year).rglob("*.parquet"):
        df = pd.read_parquet(file, columns=["Datum"])

        d = pd.to_datetime(
            df["Datum"],
            format="%d.%m.%Y",
            errors="coerce"
        )

        dates.append(d.min())
        dates.append(d.max())

print("First date:", min(dates))
print("Last date:", max(dates))

First date: 2023-01-01 00:00:00
Last date: 2026-07-31 00:00:00


In [11]:
from pathlib import Path
import pandas as pd

root = Path("../../store/parquet_data")

all_mpids = set()

for file in root.rglob("LG_AIM2Hackerdays*.parquet"):
    df = pd.read_parquet(file, columns=["MP ID"])

    mpids = pd.to_numeric(df["MP ID"], errors="coerce").dropna()
    all_mpids.update(mpids.astype(int).unique())

print("Households with time-series data:", len(all_mpids))

Households with time-series data: 92840


In [12]:
for year in ["2023", "2024", "2025", "2026"]:
    year_mpids = set()

    for file in (root / year).rglob("*.parquet"):
        try:
            df = pd.read_parquet(file, columns=["MP ID"])
            ids = pd.to_numeric(df["MP ID"], errors="coerce").dropna()
            year_mpids.update(ids.astype(int).unique())
        except Exception:
            pass

    print(year, len(year_mpids))

2023 0
2024 52845
2025 79167
2026 90675
